In [1]:
# CELL 1: IMPORTS + STATE + LLM SETUP
# =======================================

import os
os.environ["LANGCHAIN_TRACING_V2"] = "false"

from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Optional, List
from datetime import datetime
import uuid, json, asyncio, re, textwrap, nest_asyncio

# Fix asyncio trong Jupyter (cho phép nested event loops)
nest_asyncio.apply()

# === FREE TOOLS (không cần đăng ký thêm) ===
from groq import Groq
from crawl4ai import AsyncWebCrawler, CrawlerRunConfig
from bs4 import BeautifulSoup
from ddgs import DDGS  # DuckDuckGo search

# === Groq LLM (hardcoded key - test only) ===
client = Groq(api_key="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")
MODEL = "meta-llama/llama-4-scout-17b-16e-instruct"

def call_llm(prompt: str, max_tokens: int = 1200) -> str:
    """Gọi Groq LLM - free tier"""
    try:
        msg = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            max_completion_tokens=max_tokens,
            temperature=0.3,  # Thấp hơn cho output ổn định
        )
        return msg.choices[0].message.content or ""
    except Exception as e:
        print(f"⚠️ LLM error: {e}")
        return ""

# === STATE ===
class CompetitorResearchState(TypedDict):
    session_id: str
    business_name: str
    business_address: str
    industry: str
    location_query: str  # Tên địa điểm rút gọn (vd: "Quận 7", "Đà Nẵng")
    competitors: List[dict]
    all_reviews: List[dict]
    insights: List[dict]
    personas: List[dict]
    status: str
    current_agent: str
    error: Optional[str]

def init_state(name: str, address: str, industry: str) -> CompetitorResearchState:
    """Khởi tạo state từ 3 input của user"""
    # Rút gọn location từ địa chỉ (lấy phần sau cùng hoặc tên thành phố/quận)
    location = _extract_location(address)
    return {
        "session_id": str(uuid.uuid4())[:8],
        "business_name": name,
        "business_address": address,
        "industry": industry,
        "location_query": location,
        "competitors": [],
        "all_reviews": [],
        "insights": [],
        "personas": [],
        "status": "running",
        "current_agent": "agent_1_find_competitors",
        "error": None
    }

def _extract_location(address: str) -> str:
    """Tách tên quận/thành phố từ địa chỉ"""
    # Thử tìm pattern: Quận X, TP.HCM / District X, Da Nang / v.v.
    patterns = [
        r'Quận\s+\d+[^,]*',
        r'Quận\s+[A-Za-zÀ-ỹ]+[^,]*',
        r'District\s+\d+[^,]*',
        r'[Đđ]à\s+Nẵng',
        r'[Hh]ội\s+[Aa]n',
        r'[Tt]P\.?\s*[Hh][Cc][Mm]',
        r'[Hh]ồ\s+[Cc]hí\s+[Mm]inh',
        r'[Hh]à\s+[Nn]ội',
    ]
    for p in patterns:
        m = re.search(p, address, re.IGNORECASE)
        if m:
            return m.group(0)
    # Fallback: lấy phần sau cùng sau dấu phẩy
    parts = [p.strip() for p in address.split(',')]
    return parts[-1] if parts else address

print("✅ CELL 1: Imports + State + LLM loaded")
print(f"   Ví dụ location extract: '123 Nguyễn Văn A, Quận 7, TP.HCM' → '{_extract_location('123 Nguyễn Văn A, Quận 7, TP.HCM')}'")

✅ CELL 1: Imports + State + LLM loaded
   Ví dụ location extract: '123 Nguyễn Văn A, Quận 7, TP.HCM' → 'Quận 7'


In [2]:
# CELL 2: AGENT 1 - TÌM ĐỐI THỦ THẬT
# =======================================

async def _search_competitors_ddgs(business_name: str, location: str, industry: str) -> List[dict]:
    """
    Dùng DuckDuckGo search tìm đối thủ thật.
    Search query: "cafe Quận 7 review", "hotel Đà Nẵng booking", v.v.
    """
    competitors = []
    
    # Tạo query từ industry + location
    industry_clean = industry.split('/')[0].strip().lower()  # "Cafe/Specialty" → "cafe"
    queries = [
        f"{industry_clean} {location} review",
        f"{industry_clean} {location} đánh giá",
        f"{industry_clean} near {location}",
        f"best {industry_clean} {location}",
    ]
    
    seen_names = set()
    
    with DDGS() as ddgs:
        for q in queries:
            print(f"  🔍 DDGS: '{q}'")
            try:
                results = ddgs.text(q, max_results=15)
                for r in results:
                    title = r.get('title', '')
                    href = r.get('href', '')
                    body = r.get('body', '')
                    
                    # Lọc bỏ kết quả không phải đối thủ
                    if not title or len(title) < 3:
                        continue
                    if any(bad in title.lower() for bad in ['wikipedia', 'tripadvisor list', 'top 10', 'cách làm', 'hướng dẫn']):
                        continue
                    
                    # Tách tên đối thủ từ title
                    name = _clean_name_from_title(title)
                    if not name or name.lower() in seen_names:
                        continue
                    seen_names.add(name.lower())
                    
                    # Phỏng đoán rating từ snippet nếu có
                    rating = _extract_rating_from_text(body + " " + title)
                    review_count = _extract_review_count(body + " " + title)
                    
                    comp = {
                        "name": name,
                        "source_url": href,
                        "search_query": q,
                        "snippet": body[:200],
                        "rating": rating,
                        "review_count": review_count,
                        "distance_km": None,  # DDGS không có distance
                        "price_segment": _guess_price_segment(body + " " + title),
                        "scraped": False
                    }
                    competitors.append(comp)
                    
                    if len(competitors) >= 15:
                        break
                        
            except Exception as e:
                print(f"  ⚠️ DDGS error for '{q}': {e}")
                continue
            
            if len(competitors) >= 15:
                break
    
    return competitors[:15]

def _clean_name_from_title(title: str) -> str:
    """Tách tên business từ title search result"""
    # Bỏ suffix common
    title = re.sub(r'\s*[-|]\s*(Facebook|Google Maps|Tripadvisor|Booking\.com|Foody|ShopeeFood|GrabFood).*', '', title, flags=re.IGNORECASE)
    title = re.sub(r'\s*[-|]\s*.*review.*', '', title, flags=re.IGNORECASE)
    title = title.strip()
    # Bỏ nếu quá ngắn hoặc chỉ là generic
    if len(title) < 3 or title.lower() in ['home', 'menu', 'reviews', 'photos']:
        return None
    return title

def _extract_rating_from_text(text: str) -> Optional[float]:
    """Tìm rating x.x/5 trong text"""
    patterns = [
        r'(\d\.\d)\s*/\s*5',
        r'(\d\.\d)\s*⭐',
        r'rating\s*[:=]?\s*(\d\.\d)',
        r'(\d\.\d)\s*out of\s*5',
    ]
    for p in patterns:
        m = re.search(p, text, re.IGNORECASE)
        if m:
            val = float(m.group(1))
            if 1 <= val <= 5:
                return val
    return None

def _extract_review_count(text: str) -> Optional[int]:
    """Tìm số reviews trong text"""
    patterns = [
        r'(\d{1,3}(?:,\d{3})*)\s*reviews?',
        r'(\d{1,3}(?:,\d{3})*)\s*đánh giá',
        r'(\d{1,3}(?:,\d{3})*)\s*bình luận',
        r'(\d+)\s*review',
    ]
    for p in patterns:
        m = re.search(p, text, re.IGNORECASE)
        if m:
            num = m.group(1).replace(',', '')
            return int(num)
    return None

def _guess_price_segment(text: str) -> str:
    """Đoán phân khúc giá từ text"""
    text_lower = text.lower()
    if any(w in text_lower for w in ['luxury', 'cao cấp', '5 sao', 'sang trọng', 'premium', 'đắt', 'expensive']):
        return "luxury"
    elif any(w in text_lower for w in ['budget', 'rẻ', 'giá rẻ', 'bình dân', '2 sao', 'hostel']):
        return "budget"
    else:
        return "mid-range"

async def _enrich_with_crawl4ai(competitors: List[dict]) -> List[dict]:
    """
    Dùng crawl4ai vào từng source_url để lấy thêm:
    - rating chính xác hơn
    - review count
    - địa chỉ
    - menu/prices
    """
    enriched = []
    
    async with AsyncWebCrawler() as crawler:
        for comp in competitors[:8]:  # Giới hạn 8 để tránh timeout
            url = comp.get('source_url')
            if not url:
                enriched.append(comp)
                continue
            
            print(f"  🕷️ Crawl4AI: {comp['name'][:40]}...")
            
            try:
                config = CrawlerRunConfig(cache_mode="bypass", timeout=15000)
                result = await crawler.arun(url=url, config=config)
                
                if result.success and result.markdown:
                    text = result.markdown
                    
                    # Cập nhật rating nếu crawl được chính xác hơn
                    new_rating = _extract_rating_from_text(text)
                    if new_rating and (comp['rating'] is None or abs(new_rating - (comp['rating'] or 0)) > 0.3):
                        comp['rating'] = new_rating
                    
                    # Cập nhật review count
                    new_count = _extract_review_count(text)
                    if new_count and (comp['review_count'] is None or new_count > (comp['review_count'] or 0)):
                        comp['review_count'] = new_count
                    
                    # Tìm địa chỉ
                    addr = _extract_address_from_text(text)
                    if addr:
                        comp['address'] = addr
                    
                    # Tìm description
                    desc = _extract_description(text)
                    if desc:
                        comp['description'] = desc
                    
                    comp['scraped'] = True
                    comp['crawled_text'] = text[:3000]  # Lưu text để agent 2 dùng
                    
            except Exception as e:
                print(f"    ⚠️ Crawl error: {str(e)[:80]}")
            
            enriched.append(comp)
            await asyncio.sleep(0.5)  # Rate limit
    
    return enriched

def _extract_address_from_text(text: str) -> Optional[str]:
    """Tìm địa chỉ trong text"""
    # Pattern: số + đường + quận + thành phố
    patterns = [
        r'\d+[^,]{5,50}(?:Quận|District|TP\.|Thành phố)[^,]{3,30}',
        r'(?:Address|Địa chỉ)[:：]\s*([^<\n]{10,100})',
    ]
    for p in patterns:
        m = re.search(p, text, re.IGNORECASE)
        if m:
            return m.group(1) if m.lastindex else m.group(0)
    return None

def _extract_description(text: str) -> Optional[str]:
    """Tìm description/mô tả ngắn"""
    # Lấy đoạn đầu tiên có ý nghĩa (bỏ nav, menu, v.v.)
    lines = [l.strip() for l in text.split('\n') if len(l.strip()) > 30 and len(l.strip()) < 300]
    if lines:
        return lines[0]
    return None

def agent_1_find_competitors(state: CompetitorResearchState) -> CompetitorResearchState:
    """Agent 1: Tìm đối thủ thật bằng DDGS + Crawl4AI"""
    print(f"\n{'='*60}")
    print(f"🔍 AGENT 1: Tìm đối thủ cho '{state['business_name']}'")
    print(f"   Location: {state['location_query']}")
    print(f"   Industry: {state['industry']}")
    print(f"{'='*60}")
    
    try:
        # Step 1: DDGS search
        competitors = asyncio.get_event_loop().run_until_complete(
            _search_competitors_ddgs(
                state['business_name'],
                state['location_query'],
                state['industry']
            )
        )
        print(f"   📊 DDGS tìm được {len(competitors)} kết quả")
        
        # Step 2: Enrich với crawl4ai
        if competitors:
            competitors = asyncio.get_event_loop().run_until_complete(
                _enrich_with_crawl4ai(competitors)
            )
        
        # Step 3: Lọc bỏ chính business của user (nếu tìm thấy)
        filtered = []
        user_name_lower = state['business_name'].lower()
        for c in competitors:
            if user_name_lower not in c['name'].lower():
                filtered.append(c)
        
        state['competitors'] = filtered
        state['current_agent'] = 'agent_2_scrape_reviews'
        
        print(f"\n✅ AGENT 1: {len(filtered)} đối thủ thật")
        for i, c in enumerate(filtered[:5], 1):
            rating = f"{c['rating']}⭐" if c['rating'] else "N/A"
            reviews = f"{c['review_count']} reviews" if c['review_count'] else "N/A reviews"
            print(f"   {i}. {c['name'][:50]} | {rating} | {reviews} | {c['price_segment']}")
        
    except Exception as e:
        print(f"⚠️ AGENT 1 Error: {e}")
        state['error'] = str(e)
        state['competitors'] = []
    
    return state

print("✅ CELL 2: Agent 1 (Find Competitors - Real) loaded")

✅ CELL 2: Agent 1 (Find Competitors - Real) loaded


In [3]:
# CELL 3: AGENT 2 - CÀO REVIEWS THẬT
# =======================================

async def _scrape_reviews_for_competitor(comp: dict, location: str) -> List[dict]:
    """
    Cào reviews cho 1 đối thủ:
    1. Search Google Maps / review sites bằng DDGS
    2. Crawl vào trang đánh giá lấy reviews
    """
    reviews = []
    comp_name = comp['name']
    
    # Tạo search queries để tìm trang reviews
    queries = [
        f'"{comp_name}" review',
        f'"{comp_name}" đánh giá',
        f'"{comp_name}" {location} google review',
        f'"{comp_name}" foody review',
    ]
    
    review_urls = []
    
    # Tìm URLs có reviews
    with DDGS() as ddgs:
        for q in queries:
            try:
                results = ddgs.text(q, max_results=8)
                for r in results:
                    url = r.get('href', '')
                    title = r.get('title', '').lower()
                    # Ưu tiên các trang có reviews
                    if any(site in url for site in ['google.com/maps', 'tripadvisor', 'booking.com', 'food', 'review']):
                        if url not in review_urls:
                            review_urls.append(url)
                    elif 'review' in title or 'đánh giá' in title:
                        if url not in review_urls:
                            review_urls.append(url)
            except:
                continue
    
    # Crawl từng URL lấy reviews
    async with AsyncWebCrawler() as crawler:
        for url in review_urls[:5]:  # Giới hạn 5 URLs
            print(f"    🕷️ Crawl reviews: {url[:60]}...")
            
            try:
                config = CrawlerRunConfig(cache_mode="bypass", timeout=12000)
                result = await crawler.arun(url=url, config=config)
                
                if result.success and result.markdown:
                    text = result.markdown
                    # Trích xuất reviews từ text
                    new_reviews = _extract_reviews_from_text(text, comp_name, url)
                    reviews.extend(new_reviews)
                    
            except Exception as e:
                print(f"      ⚠️ {str(e)[:60]}")
            
            await asyncio.sleep(0.5)
            
            if len(reviews) >= 50:  # Đủ 50 reviews/đối thủ
                break
    
    # Nếu crawl không đủ, dùng LLM generate từ snippet đã có
    if len(reviews) < 5 and comp.get('crawled_text'):
        print(f"    🤖 LLM generate reviews từ description...")
        generated = _generate_reviews_from_description(comp)
        reviews.extend(generated)
    
    return reviews

def _extract_reviews_from_text(text: str, comp_name: str, source_url: str) -> List[dict]:
    """Trích xuất reviews từ crawled text"""
    reviews = []
    
    # Pattern 1: Tìm block review có rating + content
    # VD: "★★★★★ Great place..." hoặc "5/5 Excellent..."
    review_blocks = re.split(r'(?=(?:★{1,5}|\d\.\d\s*/\s*5|\d\s*out of\s*5))', text)
    
    for block in review_blocks[:20]:  # Giới hạn 20 blocks
        if len(block) < 30:
            continue
            
        # Tìm rating
        rating = None
        star_match = re.search(r'★{1,5}', block)
        if star_match:
            rating = len(star_match.group())
        else:
            num_match = re.search(r'(\d\.\d)\s*/\s*5', block)
            if num_match:
                rating = float(num_match.group(1))
        
        # Tìm content (đoạn text sau rating)
        content = re.sub(r'★{1,5}|\d\.\d\s*/\s*5|\d\s*out of\s*5', '', block, count=1)
        content = content.strip()
        
        # Làm sạch content
        content = re.sub(r'^[^\wÀ-ỹ]+', '', content)  # Bỏ ký tự lạ đầu dòng
        content = content[:500]  # Giới hạn độ dài
        
        if len(content) > 20 and rating:
            # Tìm reviewer name
            name_match = re.search(r'([A-Z][a-z]+\s[A-Z][a-z]+|[A-Z][a-z]+)', text[:text.find(block)+50] if block in text else text[:200])
            reviewer = name_match.group(1) if name_match else f"Guest_{len(reviews)+1}"
            
            # Tìm date
            date_match = re.search(r'(\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|\d{4}-\d{2}-\d{2})', block)
            date = date_match.group(1) if date_match else datetime.now().strftime("%Y-%m-%d")
            
            reviews.append({
                "competitor_name": comp_name,
                "reviewer_name": reviewer,
                "content": content,
                "rating": rating,
                "date": date,
                "source_url": source_url,
                "source_type": "crawled"
            })
    
    return reviews

def _generate_reviews_from_description(comp: dict) -> List[dict]:
    """Dùng LLM generate reviews từ description khi không crawl được"""
    desc = comp.get('description', '') or comp.get('snippet', '')
    name = comp['name']
    
    prompt = f"""Dựa vào mô tả sau về "{name}", viết 5 reviews giả lập thực tế (như khách hàng thật viết).
Mỗi review phải có: rating (1-5), nội dung (tiếng Việt hoặc Anh, tự nhiên), pain point hoặc praise cụ thể.

Mô tả: {desc[:500]}

Trả về JSON array:
[
  {{"rating": 5, "content": "...", "tone": "positive"}},
  {{"rating": 2, "content": "...", "tone": "negative"}}
]"""
    
    response = call_llm(prompt, max_tokens=800)
    
    try:
        # Tìm JSON trong response
        json_match = re.search(r'\[[\s\S]*\]', response)
        if json_match:
            data = json.loads(json_match.group())
            reviews = []
            for i, item in enumerate(data[:5]):
                reviews.append({
                    "competitor_name": name,
                    "reviewer_name": f"Guest_{i+1}",
                    "content": item.get('content', ''),
                    "rating": item.get('rating', 3),
                    "date": datetime.now().strftime("%Y-%m-%d"),
                    "source_type": "llm_generated",
                    "tone": item.get('tone', 'neutral')
                })
            return reviews
    except:
        pass
    
    return []

def agent_2_scrape_reviews(state: CompetitorResearchState) -> CompetitorResearchState:
    """Agent 2: Cào reviews từ đối thủ thật"""
    print(f"\n{'='*60}")
    print(f"📥 AGENT 2: Cào reviews từ {len(state['competitors'])} đối thủ")
    print(f"{'='*60}")
    
    all_reviews = []
    
    # Giới hạn 5 đối thủ đầu tiên để tránh timeout
    for i, comp in enumerate(state['competitors'][:5], 1):
        print(f"\n   [{i}/5] {comp['name'][:50]}...")
        
        try:
            reviews = asyncio.get_event_loop().run_until_complete(
                _scrape_reviews_for_competitor(comp, state['location_query'])
            )
            all_reviews.extend(reviews)
            print(f"      ✅ {len(reviews)} reviews")
            
        except Exception as e:
            print(f"      ⚠️ Error: {str(e)[:60]}")
    
    state['all_reviews'] = all_reviews
    state['current_agent'] = 'agent_3_analyze_llm'
    
    print(f"\n✅ AGENT 2: Tổng {len(all_reviews)} reviews từ {min(5, len(state['competitors']))} đối thủ")
    
    return state

print("✅ CELL 3: Agent 2 (Scrape Reviews - Real) loaded")

✅ CELL 3: Agent 2 (Scrape Reviews - Real) loaded


In [4]:
# CELL 4: AGENT 3 - LLM PHÂN TÍCH
# =======================================

def _analyze_single_review(review: dict) -> dict:
    """Phân tích 1 review bằng LLM"""
    content = review['content']
    rating = review['rating']
    comp_name = review['competitor_name']
    
    prompt = f"""Phân tích review sau về "{comp_name}":

Review: "{content}"
Rating: {rating}/5

Trích xuất thông tin và trả về JSON chính xác:
{{
  "pain_points": ["mô tả ngắn pain point 1", "pain point 2"],  // Chỉ nếu review negative hoặc mixed
  "praises": ["mô tả ngắn praise 1", "praise 2"],  // Chỉ nếu review positive hoặc mixed
  "sentiment": "positive" | "negative" | "mixed" | "neutral",
  "keywords": ["từ khóa 1", "từ khóa 2", "từ khóa 3"],  // Tối đa 5 từ khóa liên quan
  "user_type": "du khách" | "công sở" | "gia đình" | "cặp đôi" | "học sinh/sinh viên" | "freelancer" | "không rõ",
  "location_hint": "địa điểm nếu có" | null,
  "price_concern": true | false,  // Có nhắc đến giá cả không
  "service_concern": true | false,  // Có nhắc đến phục vụ không
  "cleanliness_concern": true | false,
  "location_concern": true | false
}}

Chỉ trả về JSON, không giải thích gì thêm.""" 

    response = call_llm(prompt, max_tokens=600)
    
    # Parse JSON
    try:
        json_match = re.search(r'\{[\s\S]*\}', response)
        if json_match:
            data = json.loads(json_match.group())
        else:
            data = json.loads(response)
    except:
        # Fallback nếu LLM không trả về JSON đúng
        data = _fallback_analysis(content, rating)
    
    return {
        "review_id": id(review),  # Unique ID
        "competitor_name": comp_name,
        "original_content": content[:200],
        "rating": rating,
        **data
    }

def _fallback_analysis(content: str, rating: int) -> dict:
    """Phân tích đơn giản khi LLM fail"""
    content_lower = content.lower()
    
    pain_points = []
    praises = []
    
    if rating <= 2:
        sentiment = "negative"
        if any(w in content_lower for w in ['giá', 'price', 'đắt', 'mắc', 'expensive']):
            pain_points.append("giá cao")
        if any(w in content_lower for w in ['phục vụ', 'service', 'nhân viên', 'staff']):
            pain_points.append("phục vụ kém")
        if any(w in content_lower for w in ['bẩn', 'dirty', 'sạch', 'clean']):
            pain_points.append("vệ sinh")
    elif rating >= 4:
        sentiment = "positive"
        if any(w in content_lower for w in ['sạch', 'clean', 'đẹp', 'nice']):
            praises.append("sạch sẽ")
        if any(w in content_lower for w in ['phục vụ tốt', 'good service', 'thân thiện']):
            praises.append("phục vụ tốt")
    else:
        sentiment = "mixed"
    
    # Detect user type
    user_type = "không rõ"
    if any(w in content_lower for w in ['gia đình', 'family', 'con', 'kids']):
        user_type = "gia đình"
    elif any(w in content_lower for w in ['công việc', 'work', 'laptop', 'meeting']):
        user_type = "công sở"
    elif any(w in content_lower for w in ['du lịch', 'travel', 'tour', 'visit']):
        user_type = "du khách"
    
    return {
        "pain_points": pain_points,
        "praises": praises,
        "sentiment": sentiment,
        "keywords": list(set(re.findall(r'\b\w{4,}\b', content_lower)))[:5],
        "user_type": user_type,
        "location_hint": None,
        "price_concern": any(w in content_lower for w in ['giá', 'price']),
        "service_concern": any(w in content_lower for w in ['phục vụ', 'service', 'nhân viên']),
        "cleanliness_concern": any(w in content_lower for w in ['sạch', 'clean', 'bẩn', 'dirty']),
        "location_concern": any(w in content_lower for w in ['vị trí', 'location', 'xa', 'gần']),
    }

def agent_3_analyze_llm(state: CompetitorResearchState) -> CompetitorResearchState:
    """Agent 3: Phân tích tất cả reviews bằng LLM"""
    print(f"\n{'='*60}")
    print(f"🤖 AGENT 3: Phân tích {len(state['all_reviews'])} reviews bằng LLM")
    print(f"{'='*60}")
    
    insights = []
    
    for i, review in enumerate(state['all_reviews']):
        if i % 10 == 0:
            print(f"   Analyzing {i+1}/{len(state['all_reviews'])}...")
        
        try:
            analysis = _analyze_single_review(review)
            insights.append(analysis)
        except Exception as e:
            print(f"   ⚠️ Review {i} error: {e}")
    
    state['insights'] = insights
    state['current_agent'] = 'agent_4_create_personas'
    
    # Thống kê nhanh
    sentiments = {}
    for ins in insights:
        s = ins.get('sentiment', 'unknown')
        sentiments[s] = sentiments.get(s, 0) + 1
    
    print(f"\n✅ AGENT 3: Phân tích xong {len(insights)} reviews")
    print(f"   Sentiment: {sentiments}")
    
    return state

print("✅ CELL 4: Agent 3 (LLM Analysis) loaded")

✅ CELL 4: Agent 3 (LLM Analysis) loaded


In [5]:
# CELL 5: AGENT 4 - TẠO PERSONA
# =======================================

def agent_4_create_personas(state: CompetitorResearchState) -> CompetitorResearchState:
    """Agent 4: Gom nhóm insights thành persona có tên, đặc điểm, ước tính"""
    print(f"\n{'='*60}")
    print(f"👥 AGENT 4: Tạo persona từ {len(state['insights'])} insights")
    print(f"{'='*60}")
    
    insights = state['insights']
    if not insights:
        print("⚠️ Không có insights để tạo persona")
        state['personas'] = []
        state['current_agent'] = 'agent_5_generate_content'
        return state
    
    # === BƯỚC 1: Gom nhóm thủ công theo user_type + pain_points ===
    groups = {}
    
    for ins in insights:
        user_type = ins.get('user_type', 'không rõ')
        # Tạo key từ user_type + top pain_points
        pains = tuple(sorted(ins.get('pain_points', [])[:2]))
        praises = tuple(sorted(ins.get('praises', [])[:2]))
        
        # Ưu tiên group theo pain points nếu có, không thì theo praises
        key = (user_type, pains if pains else praises)
        
        if key not in groups:
            groups[key] = []
        groups[key].append(ins)
    
    # === BƯỚC 2: Tạo persona từ mỗi nhóm ===
    personas = []
    total = len(insights)
    
    for (user_type, issues), group in groups.items():
        if len(group) < 2:  # Bỏ nhóm quá nhỏ
            continue
        
        # Tính % trong tổng
        pct = round(100 * len(group) / total, 1)
        
        # Đếm pain points phổ biến nhất
        all_pains = {}
        all_praises = {}
        all_keywords = {}
        locations = {}
        
        for ins in group:
            for p in ins.get('pain_points', []):
                all_pains[p] = all_pains.get(p, 0) + 1
            for p in ins.get('praises', []):
                all_praises[p] = all_praises.get(p, 0) + 1
            for k in ins.get('keywords', []):
                all_keywords[k] = all_keywords.get(k, 0) + 1
            loc = ins.get('location_hint')
            if loc:
                locations[loc] = locations.get(loc, 0) + 1
        
        # Top issues
        top_pains = sorted(all_pains.items(), key=lambda x: x[1], reverse=True)[:3]
        top_praises = sorted(all_praises.items(), key=lambda x: x[1], reverse=True)[:3]
        top_keywords = sorted(all_keywords.items(), key=lambda x: x[1], reverse=True)[:5]
        top_location = max(locations.items(), key=lambda x: x[1])[0] if locations else state['location_query']
        
        # Tạo tên persona bằng LLM
        persona_prompt = f"""Tạo tên người Việt Nam thực tế (1 tên) cho persona này:
- Loại khách: {user_type}
- Địa điểm: {top_location}
- Pain points chính: {[p[0] for p in top_pains]}
- Điểm khen chính: {[p[0] for p in top_praises]}

Chỉ trả về tên (vd: "Chị Linh", "Anh Minh", "Chị Hương"), không giải thích.""" 
        
        persona_name = call_llm(persona_prompt, max_tokens=50).strip().replace('"', '').replace("'", "")
        if not persona_name or len(persona_name) > 20:
            # Fallback tên
            names = {
                'công sở': 'Chị Lan', 'gia đình': 'Chị Hương', 'du khách': 'Anh Minh',
                'cặp đôi': 'Chị Linh', 'học sinh/sinh viên': 'Bạn Nam', 'freelancer': 'Anh Tú',
                'không rõ': 'Khách hàng'
            }
            persona_name = names.get(user_type, 'Khách hàng')
        
        persona = {
            "persona_id": f"p_{len(personas)+1}",
            "name": persona_name,
            "user_type": user_type,
            "location": top_location,
            "pain_points": [{"issue": p[0], "count": p[1], "frequency": f"{round(100*p[1]/len(group))}%"} for p in top_pains],
            "praises": [{"praise": p[0], "count": p[1]} for p in top_praises],
            "keywords": [k[0] for k in top_keywords],
            "estimated_count": len(group),
            "percentage": pct,
            "sample_reviews": [g['original_content'][:100] for g in group[:3]],
            "competitors_mentioned": list(set(g['competitor_name'] for g in group))
        }
        personas.append(persona)
    
    # Sắp xếp theo % giảm dần
    personas.sort(key=lambda x: x['percentage'], reverse=True)
    
    state['personas'] = personas
    state['current_agent'] = 'agent_5_generate_content'
    state['status'] = 'completed'  # Phase 1 core xong
    
    print(f"\n✅ AGENT 4: Tạo xong {len(personas)} persona")
    for p in personas:
        pains = ", ".join([x['issue'] for x in p['pain_points'][:2]])
        print(f"\n   👤 {p['name']} ({p['user_type']}) - {p['percentage']}% ({p['estimated_count']} reviews)")
        print(f"      📍 {p['location']} | 😤 Pain: {pains}")
        print(f"      🔑 Keywords: {', '.join(p['keywords'][:3])}")
    
    return state

print("✅ CELL 5: Agent 4 (Create Personas) loaded")

✅ CELL 5: Agent 4 (Create Personas) loaded


In [6]:
# CELL 6: BUILD LANGGRAPH + RUN
# =======================================

workflow = StateGraph(CompetitorResearchState)

# Add nodes
workflow.add_node("agent_1_find_competitors", agent_1_find_competitors)
workflow.add_node("agent_2_scrape_reviews", agent_2_scrape_reviews)
workflow.add_node("agent_3_analyze_llm", agent_3_analyze_llm)
workflow.add_node("agent_4_create_personas", agent_4_create_personas)

# Connect edges
workflow.add_edge(START, "agent_1_find_competitors")
workflow.add_edge("agent_1_find_competitors", "agent_2_scrape_reviews")
workflow.add_edge("agent_2_scrape_reviews", "agent_3_analyze_llm")
workflow.add_edge("agent_3_analyze_llm", "agent_4_create_personas")
workflow.add_edge("agent_4_create_personas", END)

graph = workflow.compile()

print("✅ CELL 6: LangGraph compiled")
print("   Flow: Agent 1 → Agent 2 → Agent 3 → Agent 4 → END")

# === RUN FUNCTION ===
def run_research(business_name: str, address: str, industry: str):
    """Chạy full research workflow"""
    print(f"\n{'='*70}")
    print(f"🚀 KHỞI ĐỘNG COMPETITIVE RESEARCH")
    print(f"   Business: {business_name}")
    print(f"   Address: {address}")
    print(f"   Industry: {industry}")
    print(f"{'='*70}")
    
    initial_state = init_state(business_name, address, industry)
    final_state = graph.invoke(initial_state)
    
    print(f"\n{'='*70}")
    print(f"✅ HOÀN THÀNH PHASE 1 - CORE INTELLIGENCE")
    print(f"   Session: {final_state['session_id']}")
    print(f"   Đối thủ tìm được: {len(final_state['competitors'])}")
    print(f"   Reviews cào được: {len(final_state['all_reviews'])}")
    print(f"   Insights phân tích: {len(final_state['insights'])}")
    print(f"   Persona tạo được: {len(final_state['personas'])}")
    print(f"{'='*70}\n")
    
    return final_state

print("\n✅ Sẵn sàng! Gọi: run_research('Tên', 'Địa chỉ', 'Ngành')")

✅ CELL 6: LangGraph compiled
   Flow: Agent 1 → Agent 2 → Agent 3 → Agent 4 → END

✅ Sẵn sàng! Gọi: run_research('Tên', 'Địa chỉ', 'Ngành')


In [7]:
# CELL 7: TEST
# =======================================

# Test với business của bạn
result = run_research(
    business_name="Sontras Sea Hotel",
    address="41 Hoàng Sa Road, Son Tra District, Da Nang, Vietnam",
    industry="Hotel / Beachfront Resort"
)

# Hoặc test với cafe
# result = run_research(
#     business_name="Cafe ABC",
#     address="123 Nguyễn Văn A, Quận 7, TP.HCM",
#     industry="Cafe / Specialty coffee"
# )


🚀 KHỞI ĐỘNG COMPETITIVE RESEARCH
   Business: Sontras Sea Hotel
   Address: 41 Hoàng Sa Road, Son Tra District, Da Nang, Vietnam
   Industry: Hotel / Beachfront Resort

🔍 AGENT 1: Tìm đối thủ cho 'Sontras Sea Hotel'
   Location: Vietnam
   Industry: Hotel / Beachfront Resort
  🔍 DDGS: 'hotel Vietnam review'
   📊 DDGS tìm được 15 kết quả


[INIT].... → Crawl4AI 0.8.7 

  🕷️ Crawl4AI: Beware fake hotel reviews : r/VietNam - ...
    ⚠️ Crawl error: CrawlerRunConfig.__init__() got an unexpected keyword argument 'timeout'
  🕷️ Crawl4AI: Hotel Review: Landmark81 (a Marriott Aut...
    ⚠️ Crawl error: CrawlerRunConfig.__init__() got an unexpected keyword argument 'timeout'
  🕷️ Crawl4AI: Review of Hoi An River Town Hotel in Vie...
    ⚠️ Crawl error: CrawlerRunConfig.__init__() got an unexpected keyword argument 'timeout'
  🕷️ Crawl4AI: M Hotel Da Nang Review | Luxury, Minimal...
    ⚠️ Crawl error: CrawlerRunConfig.__init__() got an unexpected keyword argument 'timeout'
  🕷️ Crawl4AI: Hotel Review: Freedom Hotel, Ho Chi Minh...
    ⚠️ Crawl error: CrawlerRunConfig.__init__() got an unexpected keyword argument 'timeout'
  🕷️ Crawl4AI: the Reverie Saigon Ho Chi Minh City Marc...
    ⚠️ Crawl error: CrawlerRunConfig.__init__() got an unexpected keyword argument 'timeout'
  🕷️ Crawl4AI: Four Seasons The Nam Hai, Vietnam, hotel...
    ⚠️ Crawl error: CrawlerRu

[INIT].... → Crawl4AI 0.8.7 

    🕷️ Crawl reviews: https://www.reddit.com/r/VietNam/comments/1j7ul4b/what_revie...
      ⚠️ CrawlerRunConfig.__init__() got an unexpected keyword argume
    🕷️ Crawl reviews: https://www.reddit.com/r/VietNam/comments/1in6bly/fake_revie...
      ⚠️ CrawlerRunConfig.__init__() got an unexpected keyword argume
    🕷️ Crawl reviews: https://www.reddit.com/r/VietNam/comments/1qr34bg/real_hotel...
      ⚠️ CrawlerRunConfig.__init__() got an unexpected keyword argume
    🕷️ Crawl reviews: https://www.reddit.com/r/VietNam/comments/1i8rtqj/beware_fak...
      ⚠️ CrawlerRunConfig.__init__() got an unexpected keyword argume
    🕷️ Crawl reviews: https://www.reddit.com/r/VietNam/comments/1i9uszt/worried_ab...
      ⚠️ CrawlerRunConfig.__init__() got an unexpected keyword argume
      ✅ 0 reviews

   [2/5] Hotel Review: Landmark81 (a Marriott Autograph Col...


...

    🕷️ Crawl reviews: https://www.hotel-scoop.com/hoi-an-river-town-hotel-vietnam/...
      ⚠️ CrawlerRunConfig.__init__() got an unexpected keyword argume
    🕷️ Crawl reviews: https://www.tripadvisor.com/Hotel_Review-g298082-d11949373-R...
      ⚠️ CrawlerRunConfig.__init__() got an unexpected keyword argume
    🕷️ Crawl reviews: https://www.tripadvisor.com/Hotel_Review-g298082-d25208008-R...
      ⚠️ CrawlerRunConfig.__init__() got an unexpected keyword argume
    🕷️ Crawl reviews: https://muctim.tuoitre.vn/khi-nao-co-diem-thi-danh-gia-nang-...
      ⚠️ CrawlerRunConfig.__init__() got an unexpected keyword argume
    🕷️ Crawl reviews: https://vietnamnet.vn/dh-quoc-gia-tphcm-cong-bo-diem-thi-dan...
      ⚠️ CrawlerRunConfig.__init__() got an unexpected keyword argume
      ✅ 0 reviews

   [4/5] M Hotel Da Nang Review | Luxury, Minimalism & Gree...


[INIT].... → Crawl4AI 0.8.7 

    🕷️ Crawl reviews: https://www.youtube.com/watch?v=QxwARP_6G8g...
      ⚠️ CrawlerRunConfig.__init__() got an unexpected keyword argume
    🕷️ Crawl reviews: https://www.walkaboutrojo.com/walkabout-rojo-blog/5gyvcroequ...
      ⚠️ CrawlerRunConfig.__init__() got an unexpected keyword argume
    🕷️ Crawl reviews: https://www.pinterest.com/pin/m-hotel-da-nang-review--715439...
      ⚠️ CrawlerRunConfig.__init__() got an unexpected keyword argume
    🕷️ Crawl reviews: https://www.tripadvisor.co.uk/Hotel_Review-g298085-d27935364...
      ⚠️ CrawlerRunConfig.__init__() got an unexpected keyword argume
    🕷️ Crawl reviews: https://www.trip.com/hotels/da-nang-hotel-detail-117011133/m...
      ⚠️ CrawlerRunConfig.__init__() got an unexpected keyword argume
      ✅ 0 reviews

   [5/5] Hotel Review: Freedom Hotel, Ho Chi Minh City – Vi...


[INIT].... → Crawl4AI 0.8.7 

    🕷️ Crawl reviews: https://www.nomadicnotes.com/freedom-hotel-ho-chi-minh-city-...
      ⚠️ CrawlerRunConfig.__init__() got an unexpected keyword argume
    🕷️ Crawl reviews: https://www.tripadvisor.com/ShowUserReviews-g293925-d3750860...
      ⚠️ CrawlerRunConfig.__init__() got an unexpected keyword argume
    🕷️ Crawl reviews: https://en.tripadvisor.com.hk/ShowUserReviews-g293925-d37508...
      ⚠️ CrawlerRunConfig.__init__() got an unexpected keyword argume
    🕷️ Crawl reviews: https://www.trip.com/hotels/ho-chi-minh-city-hotel-detail-71...
      ⚠️ CrawlerRunConfig.__init__() got an unexpected keyword argume
    🕷️ Crawl reviews: https://www.kayak.com/Ho-Chi-Minh-City-Hotels-Elios-Hotel.32...
      ⚠️ CrawlerRunConfig.__init__() got an unexpected keyword argume
      ✅ 0 reviews

✅ AGENT 2: Tổng 0 reviews từ 5 đối thủ

🤖 AGENT 3: Phân tích 0 reviews bằng LLM

✅ AGENT 3: Phân tích xong 0 reviews
   Sentiment: {}

👥 AGENT 4: Tạo persona từ 0 insights
⚠️ Không có insights để tạo p